In [ ]:
from torchrl.data import Ten

In [ ]:
from pettingzoo.classic import tictactoe_v3

In [ ]:
env = tictactoe_v3.env(render_mode="rgb_array")
env.reset(seed=42)

In [ ]:
observation, reward, termination, truncation, info = env.last()
type(observation["observation"])

In [ ]:
observation_space = env.observation_space(env.agent_selection)
observation_space

In [ ]:
import torch
from tensordict import TensorDict
from torchrl.data import TensorStorage

# Change later to use a config file or something
state_shape = (3, 3)  # Example for tic-tac-toe
action_shape = 9,  # 9 possible actions in tic-tac-toe
batch_size = 1  # For simplicity, we can start with a batch size of 1
max_nodes = 100  # Maximum number of nodes in the MCTS tree


device = "cuda" if torch.cuda.is_available() else "cpu"




In [ ]:
tree = TensorStorage(
    storage = TensorDict(
        {
            "state": torch.zeros((max_nodes, *state_shape), device=device),
            "action": torch.zeros(max_nodes, device=device, dtype=torch.uint8),
            "value": torch.zeros(max_nodes, device=device, dtype=torch.float32),
            "visit_count": torch.zeros(max_nodes, device=device, dtype=torch.uint16),
            "parent_index": torch.full((max_nodes,), -1, dtype=torch.int64, device=device),
            "children_indices": torch.full((max_nodes, 9), -1, dtype=torch.long, device=device),  # Assuming max 9 children for tic-tac-toe
        },
        batch_size=max_nodes,
    ),
    max_size=max_nodes,
)

In [ ]:
root_tensordict = TensorDict(
    {
        "state": torch.zeros(state_shape, device=device),  # Placeholder for the initial state
        "action": torch.tensor(-1, device=device),  # No action taken at the root
        "value": torch.tensor(10, device=device),  # Initial value of
        "visit_count": torch.tensor(1, device=device),  # Root node has been visited once
        "parent_index": torch.tensor(-1, device=device),  # Root node has no parent
        "children_indices": torch.full((9,), -1, dtype=torch.long, device=device),  # No children for root
    }
)

tree[0] = root_tensordict

In [ ]:
tree["value"][0]

In [ ]:
nodes = 1  # Keep track of the number of nodes in the tree

In [ ]:
from pettingzoo import AECEnv
from pettingzoo.classic import connect_four_v3
# from src.environment import build_environment


class MCTS:
    def __init__(self, max_nodes, environment: AECEnv = None, model: torch.nn.Module = None, device="cpu"):
        self.tree: TensorStorage = TensorStorage(
            storage=TensorDict(
                {
                    "state": torch.zeros((max_nodes, *state_shape), device=device, dtype=torch.bool),
                    "action": torch.zeros(max_nodes, device=device, dtype=torch.uint8),
                    "value": torch.zeros(max_nodes, device=device, dtype=torch.float32),
                    "visit_count": torch.zeros(max_nodes, device=device, dtype=torch.int16),
                    "parent_index": torch.full((max_nodes,), -1, dtype=torch.int64, device=device),
                    "children_indices": torch.full((max_nodes, 9), -1, dtype=torch.long, device=device),  # Assuming max 9 children for tic-tac-toe
                },
                batch_size=max_nodes,
            ),
            max_size=max_nodes,
        )
        self.nodes = 1
        self.environment = environment
        self.model = model
        self.device = device
        self.c_puct = 1.0

    # def calculate_ucb(self, node_index: int, total_visits: int, c_puct: float = 1.0) -> float:
    #     if self.tree["visit_count"][node_index] == 0:
    #         return float('inf')  # Prioritize unvisited nodes
    #     else:
    #         exploitation = self.tree["value"][node_index] / self.tree["visit_count"][node_index]
    #         exploration = c_puct * torch.sqrt(torch.log(total_visits) / self.tree["visit_count"][node_index])
    #         return exploitation + exploration


    def calculate_ucb_batch(self, parent_index):
        children = self.tree["children_indices"][parent_index]
        valid_mask = children != -1
        children = children[valid_mask]

        visit_counts = self.tree["visit_count"][children].float()
        values = self.tree["value"][children]

        total_visits = self.tree["visit_count"][parent_index].float()

        # Avoid division by zero
        exploitation = torch.zeros_like(values)
        visited_mask = visit_counts > 0
        exploitation[visited_mask] = values[visited_mask] / visit_counts[visited_mask]

        exploration = torch.zeros_like(values)
        exploration[visited_mask] = self.c_puct * torch.sqrt(
            torch.log(total_visits) / visit_counts[visited_mask]
        )

        # Unvisited = +inf
        ucb = exploitation + exploration
        ucb[~visited_mask] = float("inf")

        return children, ucb

    def add_node(self, parent_index: int, state: torch.Tensor, action: int, value: float):
        if self.nodes >= self.tree.max_size:
            raise Exception("Maximum number of nodes reached")
        
        new_index = self.nodes
        self.tree["state"][new_index] = state
        self.tree["action"][new_index] = action
        self.tree["value"][new_index] = value
        self.tree["visit_count"][new_index] = 0  # Initialize visit count to 0
        self.tree["parent_index"][new_index] = parent_index
        
        if parent_index != -1:
            # Update the parent's children indices
            for i in range(9):  # Assuming max 9 children for tic-tac-toe
                if self.tree["children_indices"][parent_index][i] == -1:
                    self.tree["children_indices"][parent_index][i] = new_index
                    break
        
        self.nodes += 1

    def backpropagate(self, node_index: int, value: float):
        # Backpropagate the value from the given node index to the root
        while node_index != -1:
            self.tree["value"][node_index] += value
            self.tree["visit_count"][node_index] += 1
            node_index = self.tree["parent_index"][node_index]

    def selection(self, node_index: int, c_puct: float = 1.0) -> int:
        if self.tree["visit_count"][node_index] == 0:
            return node_index  # If the node has not been visited, select it
        
        children = self.tree["children_indices"][node_index]
        best_child_internal_index = torch.argmax(self.calculate_ucb_batch(node_index))
        best_child_index = children[best_child_internal_index]
        return self.selection(best_child_index)


In [ ]:
mcts = MCTS(max_nodes=100)


In [ ]:
print(mcts.nodes)
print(mcts.tree["children_indices"][0])

In [ ]:
mcts.add_node(parent_index=0, state=torch.zeros(state_shape, device=device), action=0, value=0.5)

In [ ]:
print(mcts.nodes)

In [ ]:
mcts.calculate_ucb_batch(0)

In [ ]:
mcts.add_node(
    parent_index=0,  # Assuming the root node is at index 0
    state=torch.zeros(state_shape, device=device),  # Placeholder for the new state
    action=torch.tensor(0, device=device),  # Example action
    value=torch.tensor(0.0, device=device)  # Example value
)

In [ ]:
print(mcts.nodes)
print(mcts.tree["children_indices"][0])
print(mcts.tree["parent_index"][1])

In [ ]:
mcts.backpropagate(node_index=1, value=3.0)  

In [ ]:
print(mcts.tree["value"][1])
print(mcts.tree["visit_count"][1])

print(mcts.tree["value"][0])
print(mcts.tree["visit_count"][0])

In [ ]:
def model_predict(state):
    # Placeholder for model prediction logic
    # In a real implementation, this would use a trained model to predict the value and policy
    value = torch.rand(1)  # Random value prediction
    policy = torch.rand(9)  # Random policy prediction for 9 actions
    return value, policy


In [ ]:
def add_node(tree, parent_index, state, action, value):
    